# 5. Preprocesamiento
MinDefensa publica conteos diarios. La unidad de pronóstico es una fecha en Bogotá. Consultar `data/README.md` para descargar el CSV. El Excel nacional de 2025 en DataSet tiene otra granularidad y cobertura, por lo que se conserva como material de consulta.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# En Colab, sitúe el directorio de trabajo en la raíz del repositorio.
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
raw = root / 'data' / 'raw'
processed = root / 'data' / 'processed'
processed.mkdir(parents=True, exist_ok=True)

def input_path(name):
    for folder in [root / 'DataSet', raw]:
        path = folder / name
        if path.is_file():
            return path
    return raw / name

In [ ]:
import json
import numpy as np

path = input_path('hurto_bogota.csv')
if not path.is_file():
    raise FileNotFoundError(f'Descargue {path.name} según data/README.md')

source = pd.read_csv(path, dtype={'cod_muni': 'string'})
display(pd.DataFrame({'tipo_original': source.dtypes.astype(str), 'nulos': source.isna().sum()}))
national = input_path('hurto_nacional_filas.json')
if national.is_file():
    print('Filas del portal nacional (toda su cobertura):', json.loads(national.read_text())[0]['count'])
print('Archivo:', path, '| Filas leídas:', len(source))
print('Duplicados exactos en el archivo:', source.duplicated().sum())

dates = pd.to_datetime(source['fecha_hecho'], format='ISO8601', errors='coerce').dt.normalize()
city = source['cod_muni'].str.strip().eq('11001').fillna(False)
period = pd.date_range('2018-01-01', '2025-12-31', name='fecha')
selected = city & dates.between(period[0], period[-1])
daily = source.loc[selected, ['cantidad']].assign(fecha=dates[selected])
daily['cantidad'] = pd.to_numeric(daily['cantidad'], errors='coerce')
counts = daily['cantidad']
invalid = ~np.isfinite(counts) | counts.lt(0) | counts.mod(1).ne(0)
duplicate_dates = daily['fecha'].duplicated(keep=False)
missing = period.difference(daily['fecha'])
print('Fechas inválidas en archivo:', dates.isna().sum(), '| En Bogotá:', (city & dates.isna()).sum())
print('Códigos municipales faltantes:', source['cod_muni'].isna().sum())
print('Filas filtradas:', len(daily), '| Fechas únicas:', daily['fecha'].nunique())
print('Filas con fecha duplicada:', duplicate_dates.sum(), '| Conteos inválidos:', invalid.sum())
print('Fechas faltantes:', len(missing), missing.strftime('%Y-%m-%d').tolist())
if duplicate_dates.any() or invalid.any():
    display(daily.loc[duplicate_dates | invalid].head(20))
if dates[city].isna().any() or source['cod_muni'].isna().any() or duplicate_dates.any() or invalid.any() or len(missing):
    raise ValueError('Resolver inconsistencias con la fuente antes de generar atributos diarios.')
daily = daily.set_index('fecha').sort_index()
daily['cantidad'] = daily['cantidad'].astype('int64')
observed = pd.Series({'filas': len(daily), 'fechas': daily.index.nunique(), 'victimas': daily['cantidad'].sum()})
reference = pd.Series({'filas': 2922, 'fechas': 2922, 'victimas': 975793})
display(pd.DataFrame({'observado': observed, 'referencia': reference, 'diferencia': observed - reference}))

Se exige una observación válida por fecha para que un desplazamiento de una fila equivalga a un día. Los huecos y duplicados se reportan para resolución con la fuente. Los picos se marcan con el percentil 99 de 2018–2024 como criterio descriptivo y se conservan. La marca queda fuera de los predictores.

In [ ]:
threshold = daily.loc[:'2024-12-31', 'cantidad'].quantile(.99)
daily['pico_revision'] = daily['cantidad'].gt(threshold)
print('Umbral p99:', threshold, '| Días marcados:', daily['pico_revision'].sum())
display(daily.nlargest(5, 'cantidad'))

daily['mes'] = daily.index.month
daily['dia_semana'] = daily.index.dayofweek
daily['fin_semana'] = daily['dia_semana'].ge(5)
for lag in [1, 7, 14]:
    daily[f'rezago_{lag}'] = daily['cantidad'].shift(lag)
daily['media_previa_7'] = daily['cantidad'].shift(1).rolling(7).mean()
display(daily.isna().sum().rename('nulos_por_columna'))
daily.to_csv(processed / 'hurto_diario.csv')

Cada fila tiene como objetivo la cantidad del día indicado. Los primeros 1, 7 y 14 días carecen del histórico necesario para el rezago correspondiente. La media de los siete días anteriores también queda vacía al inicio durante siete días. Estos vacíos son esperados y se conservan. El conjunto común para evaluar los tres comparadores exige los 14 días de historia.